# Markout analysis — per-client flow toxicity
For every client fill, the **markout at horizon h** is the client-signed mid move after the fill:
$$m_h = s \cdot (mid_{t+h} - mid_t), \quad s = +1\ \text{for BUY}, -1\ \text{for SELL}$$
- **Positive** markout: the market moved the client's way after they traded — informed / toxic flow; the counterparty (us, when internalized) is adversely selected.
- **Negative** markout: price mean-reverted against the client's trade — benign / noise flow; inventory taken from them tends to profit (the theoretical basis of positive drift P&L).

Horizons: 30s / 2min / 10min, share-weighted per client. Fills whose horizon extends past the last quote are dropped at that horizon (no end-of-day clamping bias). One engineered day — treat rankings as illustrative, not statistically significant.

In [ ]:
import sys
from bisect import bisect_right
from datetime import datetime, timedelta

import pandas as pd

sys.path.insert(0, '.')
import config
from internalizer.data import load_quotes

quotes = load_quotes(config.QUOTES_CSV)
qts = [q.ts for q in quotes]
mids = [(q.bid + q.ask) / 2 for q in quotes]
LAST = qts[-1]

def mid_at(ts):
    """Prevailing mid (cents) at or before ts."""
    return mids[bisect_right(qts, ts) - 1]

fills = pd.read_csv(config.FILLS_CSV, parse_dates=['timestamp'])
print(f'{len(fills)} fills, {fills.client_id.nunique()} clients')
fills.head(3)

In [ ]:
HORIZONS = {'30s': 30, '2min': 120, '10min': 600}

rows = []
for _, f in fills.iterrows():
    s = 1 if f['side'] == 'BUY' else -1          # client-signed direction
    m0 = mid_at(f['timestamp'])                  # mid at fill time
    row = {'client': f['client_id'].replace('CLIENT_', ''), 'qty': f['quantity'],
           'venue': f['venue']}
    for name, secs in HORIZONS.items():
        t1 = f['timestamp'] + timedelta(seconds=secs)
        # drop fills whose horizon runs past the tape (avoids clamping bias)
        row[name] = s * (mid_at(t1) - m0) if t1 <= LAST else None
    rows.append(row)
mo = pd.DataFrame(rows)

def share_weighted(g):
    out = {'fills': len(g), 'shares': g['qty'].sum()}
    for h in HORIZONS:
        v = g.dropna(subset=[h])
        out[f'markout {h} (c/sh)'] = round((v[h] * v['qty']).sum() / v['qty'].sum(), 2)
    return pd.Series(out)

per_client = (mo.groupby('client').apply(share_weighted, include_groups=False)
                .sort_values('markout 10min (c/sh)', ascending=False))
per_client   # positive at the top = most toxic (market moves their way after they trade)

In [ ]:
# aggregate flow character + internalized-only view (what the firm actually absorbed)
total = share_weighted(mo)
internal = share_weighted(mo[mo.venue == 'INTERNAL'])
pd.DataFrame({'all fills': total, 'internalized only': internal})

**How to read this for the strategy:**
- Clients at the top (positive 10-min markout) are candidates for a **toxicity multiplier** on the 2¢ internalization threshold, or for routing outright — internalizing their flow means holding inventory that trends against us.
- Clients at the bottom (negative markout) are the benign flow whose mean reversion is the theoretical source of positive inventory-drift P&L.
- If `internalized only` markouts are worse than `all fills`, our acceptance rule is adversely selecting toxic flow into the book; if better, the spread/coverage gates are already filtering well.
- Caveat: one engineered day, a handful of fills per client — production tiering needs weeks of data and horizon-matched significance tests.